In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from xgboost import XGBRegressor
import matplotlib.pyplot as plt
import optuna
from sklearn.model_selection import cross_val_score
from category_encoders import TargetEncoder
from sklearn.preprocessing import LabelEncoder
import seaborn as sns

C:\Users\Sofia\AppData\Local\Programs\Python\Python314\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
df = pd.read_csv('data/drom_archive_cleaned_2018-2025newbig2.csv')
df_cheap = df[df['Цена'] <= 700_000]
df_mid = df[(df['Цена'] >= 500_000) & (df['Цена'] < 2_000_000)]
df_exp = df[df['Цена'] >= 2_000_000]

In [3]:
df_mid = df_mid.drop(columns=[
    'Название машины',
    'Месяц объявления',
    'Руль'
])


In [4]:
categorical_target = ['Метка', 'Город', 'Поколение']      # Target Encoding
categorical_onehot = ['Тип двигателя', 'Коробка передач', 'Привод', 'Рестайлинг', 'Тип кузова']  # OneHot
numerical = ['Год', 'Объем двигателя', 'Мощность', 'Пробег', 'Возраст авто']
categories = ['Название машины', 'Тип двигателя', 'Коробка передач', 'Привод', 'Тип кузова', 'Метка', 'Город', 'Руль']

In [5]:
preprocessor = ColumnTransformer(
    transformers=[
        ('target', TargetEncoder(), categorical_target),
        ('onehot', OneHotEncoder(handle_unknown='ignore'), categorical_onehot),
        ('num', 'passthrough', numerical)
    ]
)

In [6]:
y = df_mid['Цена']
X = df_mid.drop('Цена', axis=1)

X_train = X[X['Год объявления'] <= 2023]
X_test = X[X['Год объявления'] >= 2024]

y_train = y[X['Год объявления'] <= 2023]
y_test = y[X['Год объявления'] >= 2024]
df_mid = df.drop(columns=['Год объявления'])

In [7]:
model = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', XGBRegressor(
        n_estimators=2000,
        n_jobs=-1,
        random_state=42,
        eval_metric='rmse'
    ))
])
model.fit(X_train, y_train, regressor__verbose=True)
y_pred = model.predict(X_test)

In [8]:
xg_mse = mean_squared_error(y_test, y_pred)
xg_rmse = np.sqrt(xg_mse)
xg_mae = mean_absolute_error(y_test, y_pred)
xg_r2 = r2_score(y_test, y_pred)

In [9]:
pd.options.display.float_format = '{:_.2f}'.format
pd.DataFrame({
    'Метод оценки': ['Среднеквадратическая ошибка (MSE)', 'Среднеквадратическая ошибка (RMSE)', 'Средняя абсолютная ошибка (MAE)', 'Коэффицент детерминации (R^2)'],
    'Результаты': [xg_mse, xg_rmse, xg_mae, xg_r2]
})

,Метод оценки,Результаты
0,Среднеквадратическая ошибка (MSE),40_393_059_008.41
1,Среднеквадратическая ошибка (RMSE),200_980.25
2,Средняя абсолютная ошибка (MAE),150_240.58
3,Коэффицент детерминации (R^2),0.74


In [11]:
from sklearn.metrics import mean_absolute_percentage_error

mape = mean_absolute_percentage_error(y_test, y_pred)
print(f"MAPE: {mape:.2%}")

MAPE: 14.42%


In [10]:
from sklearn.inspection import permutation_importance

result = permutation_importance(
    model,
    X_test,
    y_test,
    n_repeats=5,
    random_state=42,
    n_jobs=-1
)

importance = pd.Series(result.importances_mean, index=X_test.columns)
print(importance.sort_values(ascending=False))

Год               1.89
Возраст авто      0.30
Мощность          0.26
Метка             0.23
Объем двигателя   0.22
Тип кузова        0.17
Поколение         0.06
Пробег            0.06
Коробка передач   0.02
Привод            0.02
Город             0.01
Тип двигателя     0.01
Рестайлинг        0.00
Год объявления    0.00
dtype: float64
